# LAB 02 — N-gram Language Models

Kết quả được tính trên 10.000 documents đầu tiên. Cách preprocessing, chia tập và tính perplexity được đặt theo notebook tham khảo.

In [ ]:
from __future__ import annotations

import csv
import gc
import gzip
import json
import math
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_lab_directory():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        if base.name == "lab2" and (base / "ngram_lm.py").is_file():
            return base
        candidate = base / "lab2"
        if (candidate / "ngram_lm.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from lab2 or its parent folder")


LAB_DIR = find_lab_directory()
REPO_ROOT = LAB_DIR.parent
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from ngram_lm import (
    NGramLanguageModel, build_vocabulary, document_to_sentences,
    tokenize_sentence, train_bigram, train_trigram, train_unigram,
)

DATA_PATH = REPO_ROOT / "lab01" / "data" / "c4-train.00000-of-01024-30K.json.gz"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Corpus not found: {DATA_PATH}")

MAX_DOCUMENTS = 10_000
RANDOM_SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
RESULTS_PATH = LAB_DIR / "results.csv"

RESULT_COLUMNS = [
    "record_type", "experiment", "model", "n", "smoothing", "split",
    "context", "candidate", "rank", "prediction", "actual_next_word",
    "probability", "perplexity", "notes",
]
all_result_rows = []


def add_result(record_type, experiment, model="", n="", smoothing="", split="",
               context="", candidate="", rank="", prediction="",
               actual_next_word="", probability="", perplexity="", notes=""):
    all_result_rows.append({
        "record_type": record_type, "experiment": experiment, "model": model,
        "n": n, "smoothing": smoothing, "split": split, "context": context,
        "candidate": candidate, "rank": rank, "prediction": prediction,
        "actual_next_word": actual_next_word, "probability": probability,
        "perplexity": perplexity, "notes": notes,
    })

print(f"Corpus: {DATA_PATH}")
print(f"Documents: first {MAX_DOCUMENTS:,}; seed={RANDOM_SEED}; split=70/15/15")


## 13. Experiment 1 — Corpus statistics

Đếm unigram, bigram và trigram riêng trong từng câu; tần suất được nhóm theo số lần xuất hiện.

In [2]:
def load_documents(path, limit):
    documents = []
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        for line in stream:
            if len(documents) >= limit:
                break
            if not line.strip():
                continue
            record = json.loads(line)
            text = record.get("text", "") if isinstance(record, dict) else ""
            documents.append(document_to_sentences(text if isinstance(text, str) else ""))
    return documents


def flatten_documents(items):
    return [sentence for document in items for sentence in document]


documents = load_documents(DATA_PATH, MAX_DOCUMENTS)
all_sentences = flatten_documents(documents)
indices = np.random.default_rng(RANDOM_SEED).permutation(len(documents))
train_end = int(TRAIN_FRACTION * len(documents))
validation_end = int((TRAIN_FRACTION + VALIDATION_FRACTION) * len(documents))
train_documents = [documents[i] for i in indices[:train_end]]
validation_documents = [documents[i] for i in indices[train_end:validation_end]]
test_documents = [documents[i] for i in indices[validation_end:]]
train_sentences = flatten_documents(train_documents)
validation_sentences = flatten_documents(validation_documents)
test_sentences = flatten_documents(test_documents)

print(f"Documents: {len(documents):,} total | {len(train_documents):,} train | {len(validation_documents):,} validation | {len(test_documents):,} test")
print(f"Sentences: {len(all_sentences):,}")
print(f"Tokens: {sum(map(len, all_sentences)):,}")
print(f"Train/validation/test sentences: {len(train_sentences):,} / {len(validation_sentences):,} / {len(test_sentences):,}")

Documents: 10,000 total | 7,000 train | 1,500 validation | 1,500 test
Sentences: 207,954
Tokens: 3,634,369
Train/validation/test sentences: 145,151 / 31,255 / 31,548


### Corpus counts and frequency distribution

Plot trục x là số lần xuất hiện; trục y là số loại n-gram có tần suất đó.

In [ ]:
corpus_vocabulary = build_vocabulary(all_sentences, include_special_tokens=False)
ngram_functions = {1: train_unigram, 2: train_bigram, 3: train_trigram}
ngram_summary = {}
frequency_distributions = {}

for n, count_ngrams in ngram_functions.items():
    counts = count_ngrams(all_sentences)
    frequency_distributions[n] = Counter(counts.values())
    ngram_summary[n] = {
        "unique": len(counts),
        "singletons": sum(value == 1 for value in counts.values()),
    }
    del counts
    gc.collect()

print("Corpus statistics")
print(f"Documents: {len(documents):,}")
print(f"Sentences: {len(all_sentences):,}")
print(f"Tokens: {sum(map(len, all_sentences)):,}")
print(f"Vocabulary: {len(corpus_vocabulary):,}")
print("Model       Unique n-grams    Singletons")
for n, label in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
    row = ngram_summary[n]
    print(f"{label:<10} {row['unique']:>14,} {row['singletons']:>13,}")

print("\nOccurrence-frequency tables (top 10)")
for n, label in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
    table = pd.DataFrame(
        sorted(frequency_distributions[n].items())[:10],
        columns=["Occurrence count", "Number of n-grams"],
    )
    print(f"{label}:")
    display(table)


In [ ]:
def show_frequency_plot(distributions):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    colors = {1: "#2563eb", 2: "#ea580c", 3: "#16a34a"}

    for axis, (n, label) in zip(axes, ((1, "Unigram"), (2, "Bigram"), (3, "Trigram"))):
        entries = sorted(distributions[n].items())
        occurrence_counts = [item[0] for item in entries]
        number_of_ngrams = [item[1] for item in entries]
        axis.plot(occurrence_counts, number_of_ngrams, color=colors[n], linewidth=1.5)
        axis.set_xscale("log")
        axis.set_yscale("log")
        axis.set_title(f"{label} frequency distribution")
        axis.set_xlabel("Occurrence count")
        axis.set_ylabel("Number of n-grams")
        axis.grid(True, which="both", alpha=0.2)

    fig.tight_layout()
    plt.show()


show_frequency_plot(frequency_distributions)


### Discussion and observations

- **Corpus and vocabulary:** 10,000 documents produced 207,954 sentences and 3,634,369 tokens. There are 102,718 distinct words.
- **Growth in unique n-grams:** unique counts increase from 102,718 unigrams to 1,186,446 bigrams and 2,391,055 trigrams. Longer sequences distinguish more contexts, so many more types appear.
- **Sparsity:** singletons rise from 47,634 unigrams (46.37%) to 873,587 bigrams (73.63%) and 2,114,320 trigrams (88.43%). Most observed trigrams occur only once, which explains why higher-order models have less repeated evidence.
- **Frequency plot:** the distributions have a long tail: many n-gram types occur rarely, while relatively few occur often. The log scales make both ends visible.


## 16. Experiment 2 — MLE vs Laplace smoothing

Chấm bigram và trigram trên train, validation, test; vocabulary chỉ lấy từ train.

In [5]:
splits = {
    "train": train_sentences,
    "validation": validation_sentences,
    "test": test_sentences,
}
trained_models = {}
experiment2_results = []

for n, label in ((2, "Bigram"), (3, "Trigram")):
    model = NGramLanguageModel(n=n, smoothing="mle").fit(train_sentences)
    trained_models[n] = model
    for smoothing in ("mle", "laplace"):
        model.smoothing = smoothing
        scores = {split: model.perplexity(sentences) for split, sentences in splits.items()}
        experiment2_results.append({
            "Model": f"{label} {smoothing.title()}",
            "Train_PPL": scores["train"],
            "Valid_PPL": scores["validation"],
            "Test_PPL": scores["test"],
        })
        for split, score in scores.items():
            add_result("perplexity", "Experiment 2", label, n, smoothing, split, perplexity=score)
    model.smoothing = "mle"

print("Model                 Train PPL       Valid PPL        Test PPL")
for row in experiment2_results:
    print(f"{row['Model']:<20} {row['Train_PPL']:>12.4f} {row['Valid_PPL']:>15.4f} {row['Test_PPL']:>15.4f}")

Model                 Train PPL       Valid PPL        Test PPL
Bigram Mle               133.7667             inf             inf
Bigram Laplace          5933.0691       8553.3568       8688.7404
Trigram Mle               10.4365             inf             inf
Trigram Laplace        20681.1928      36847.8795      37053.1090


## 17. Perplexity

Perplexity = exp(− tổng log P(token | context) / số token). EOS không được tính vào mẫu số, giống notebook tham khảo.

### Discussion and observations

- **MLE and zero probability:** MLE gives probability zero to an n-gram absent from training. Validation and test perplexity are therefore infinite for both bigram and trigram MLE. The coverage results support this: about 33.5% of validation/test bigrams and 74.0% of trigrams are unseen.
- **Laplace smoothing:** Laplace gives unseen n-grams nonzero probability, so all its perplexities are finite. However, adding one spreads probability across a large vocabulary: training perplexity increases from 133.77 to 5,933.07 for bigram and from 10.44 to 20,681.19 for trigram.
- **Validation versus test:** test perplexity is slightly higher than validation for both smoothed models: 8,688.74 vs 8,553.36 for bigram, and 37,053.11 vs 36,847.88 for trigram.
- **Bigram versus trigram:** Laplace bigram has lower perplexity than Laplace trigram on every split. The trigram model has sparser contexts, and add-one smoothing distributes probability over many more possible continuations.
- **Interpretation:** within this 10k corpus and this implementation, Laplace avoids infinite scores but does not make the trigram model perform better than bigram. The finite perplexities remain high, so smoothing solves the zero-probability issue without removing the data sparsity.


## 19. Experiment 3 — Perplexity theo n-gram order

So sánh unigram, bigram, trigram bằng MLE trên cùng ba tập. Hai model bigram/trigram được dùng lại từ Experiment 2.

In [ ]:
trained_models[1] = NGramLanguageModel(n=1, smoothing="mle").fit(train_sentences)
experiment3_results = []

for n, label in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
    model = trained_models[n]
    model.smoothing = "mle"
    scores = {split: model.perplexity(sentences) for split, sentences in splits.items()}
    experiment3_results.append({
        "Model": label,
        "Train_PPL": scores["train"],
        "Validation_PPL": scores["validation"],
        "Test_PPL": scores["test"],
    })
    for split, score in scores.items():
        add_result("perplexity", "Experiment 3", label, n, "mle", split, perplexity=score)

print("Model       Train PPL       Validation PPL       Test PPL")
for row in experiment3_results:
    print(f"{row['Model']:<10} {row['Train_PPL']:>12.4f} {row['Validation_PPL']:>18.4f} {row['Test_PPL']:>14.4f}")

plt.figure(figsize=(8, 4.5))
plt.bar(
    [row["Model"] for row in experiment3_results],
    [row["Train_PPL"] for row in experiment3_results],
    color="#4682b4",
)
plt.title("Training perplexity by model")
plt.xlabel("Model")
plt.ylabel("Perplexity")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


### Discussion and observations

- **Training perplexity:** MLE perplexity decreases as n grows: 2,007.76 for unigram, 133.77 for bigram, and 10.44 for trigram. A longer context fits the training sequences more specifically.
- **Generalization:** validation and test perplexity are infinite for all three models. Training improvement therefore does not translate into finite held-out perplexity in this run.
- **Sparsity explains the gap:** unseen held-out rates rise with n: roughly 2.9% for unigrams, 33.4% for bigrams, and 74.0% for trigrams. Even one zero-probability token makes the sequence-set perplexity infinite under MLE.
- **Conclusion:** these results are consistent with increasing sparsity and weak generalization as context length grows. Training perplexity alone is not enough to select a model; the held-out scores and unseen n-gram rates must also be considered.


## 20. Application — Next-word prediction

Dùng MLE trigram; nếu không có kết quả thì dùng bigram. Bảng gồm top 3 từ và actual next word trong test set.

In [ ]:
def predict_next_words(context, k=3):
    predictions = trained_models[3].next_word_distribution(context)
    if not predictions:
        predictions = trained_models[2].next_word_distribution(context)
    return list(predictions.items())[:k]


def find_actual_next_word(context, sentences):
    context_tokens = tokenize_sentence(context)
    for sentence in sentences:
        for start in range(len(sentence) - len(context_tokens)):
            if sentence[start:start + len(context_tokens)] == context_tokens:
                return sentence[start + len(context_tokens)]
    return "Unknown"


contexts = ["the cat", "natural language", "machine learning", "deep learning", "computer vision"]
prediction_rows = []
for context in contexts:
    top_words = predict_next_words(context)
    actual = find_actual_next_word(context, test_sentences)
    text = ", ".join(f"{word} ({probability:.4f})" for word, probability in top_words) or "No prediction"
    print(f"{context:<20} {text:<65} actual={actual}")
    for rank, (word, probability) in enumerate(top_words, start=1):
        row = {"context": context, "candidate": word, "rank": rank,
               "prediction": top_words[0][0], "actual_next_word": actual,
               "probability": probability}
        prediction_rows.append(row)
        add_result("next_word", "Application 1", "Trigram MLE", 3, "mle",
                   "test", context, word, rank, top_words[0][0], actual, probability)

chart_predictions = predict_next_words("the cat")
if chart_predictions:
    plt.figure(figsize=(7, 4))
    plt.bar(
        [word for word, _ in chart_predictions],
        [probability for _, probability in chart_predictions],
        color="#4682b4",
    )
    plt.title("Top next words after the cat")
    plt.xlabel("Next word")
    plt.ylabel("Probability")
    plt.tight_layout()
    plt.show()


### Discussion

- The model's top prediction is house after the cat, while the first matching test-set continuation is queen; it does not match that example. For machine learning, the top prediction is and, while the first matching test continuation is has.
- The other three contexts have Unknown as the actual next word because no matching context was found in the test sentences. Their predictions cannot be checked against this lookup.
- Thus, only two of the five contexts can be compared here, and neither top prediction matches the first test occurrence. This is a small illustrative check, not a reliable accuracy estimate: a context can have multiple valid continuations, and this code records only the first match in the test set.


## 21. Application — Sentence ranking

Xếp hạng các phần tiếp nối theo log probability của trigram MLE.

In [8]:
ranking_context = ["machine", "learning"]
context_tuple = tuple(ranking_context)
continuation_counts = Counter()
for sentence in train_sentences:
    for start in range(len(sentence) - len(context_tuple)):
        if tuple(sentence[start:start + len(context_tuple)]) == context_tuple:
            continuation = sentence[start + len(context_tuple):start + len(context_tuple) + 3]
            if continuation:
                continuation_counts[" ".join(continuation)] += 1

candidates = [text for text, _ in continuation_counts.most_common(3)]
if not candidates:
    candidates = ["is useful for", "models are used", "works very well"]


def score_continuation(model, context, text):
    history = list(context)
    score = 0.0
    for word in tokenize_sentence(text):
        probability = model.probability(history, word)
        if probability == 0:
            return -math.inf
        score += math.log(probability)
        history.append(word)
    return score


ranking_rows = [(text, score_continuation(trained_models[3], ranking_context, text))
                for text in candidates]
ranking_rows.sort(key=lambda row: row[1], reverse=True)
for rank, (text, score) in enumerate(ranking_rows, start=1):
    print(f"{rank}. {text} | log P={score:.6f}")
    add_result("sentence_ranking", "Application 2", "Trigram MLE", 3, "mle",
               "train", " ".join(ranking_context), text, rank, notes=f"log_probability={score}")

1. models | log P=-1.871802
2. and sandboxing to | log P=-5.232178
3. technology and is | log P=-7.853216


### Discussion

- The highest-ranked continuation is models with log probability −1.8718. The next candidates score −5.2322 and −7.8532; because these are log probabilities, the value closer to zero is more likely under the trigram model.
- These scores compare only the three candidates generated from training continuations. They show the model's ranking preference, not whether a continuation is objectively correct or fluent.


## 22. Error analysis

Hoàn thiện phần phân tích bằng nhận xét của bạn.

## 23. Thí nghiệm context length

Bảng unseen n-gram cho unigram, bigram và trigram trên cùng các split.

In [9]:
print("Model       Split        Unseen       Total      Unseen rate")
for n, label in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
    model = trained_models[n]
    for split, sentences in splits.items():
        coverage = model.ngram_coverage(sentences)
        print(f"{label:<10} {split:<10} {coverage['unseen_ngrams']:>10,} {coverage['total_ngrams']:>11,} {coverage['unseen_rate']:.4%}")
        add_result("ngram_coverage", "Experiment 23", label, n, "mle", split,
                   probability=coverage["unseen_rate"],
                   notes=f"unseen={coverage['unseen_ngrams']}; total={coverage['total_ngrams']}")

with RESULTS_PATH.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=RESULT_COLUMNS)
    writer.writeheader()
    writer.writerows(all_result_rows)
print(f"Saved {len(all_result_rows)} rows to {RESULTS_PATH}")

Model       Split        Unseen       Total      Unseen rate
Unigram    train               0   2,524,517 0.0000%
Unigram    validation     16,208     554,270 2.9242%
Unigram    test           15,887     555,582 2.8595%
Bigram     train               0   2,379,366 0.0000%
Bigram     validation    175,036     523,015 33.4667%
Bigram     test          174,862     524,034 33.3684%
Trigram    train               0   2,237,193 0.0000%
Trigram    validation    364,843     492,265 74.1152%
Trigram    test          365,087     493,035 74.0489%
Saved 46 rows to D:\NLP_exercise\lab2\results.csv


## 24. Reflection

Hoàn thiện phần reflection bằng câu trả lời của bạn.

## 25. AI assistance statement

Cập nhật khai báo sử dụng AI theo đúng đóng góp thực tế trước khi nộp.